In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'SAS': ['Emanuel Miller'], 'DEN': ['Zeke Nnaji']}

Out Players:
{'ATL': ['Jock Landale'], 'DET': ['Caris LeVert', 'Tobias Harris', 'Duncan Robinson', 'Cade Cunningham', 'Isaiah Stewart'], 'ORL': ['Franz Wagner', 'Jonathan Isaac', 'Jett Howard'], 'PHI': ['Cameron Payne', 'Johni Broome'], 'CLE': ['James Harden', 'Dean Wade', 'Jaylon Tyson', 'Donovan Mitchell', 'Max Strus', 'Thomas Bryant'], 'MEM': ['Jahmai Mashack', 'Javon Small', 'Ty Jerome'], 'POR': ['Shaedon Sharpe', 'Jerami Grant', 'Vít Krejčí'], 'DEN': ['Peyton Watson', 'Spencer Jones']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 10 teams with confirmed lineups
Updated 2 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
150,NaN,2025-26,1643253,Toby Okani,Toby,1610612763,MEM,Memphis Grizzlies,22501135,2026-04-05T00:00:00,MEM @ MIL,L,41.166667,4,12,0.333,1,4,0.25,0,2,0.000,2,1,3,0,1,1,0,1,4,2,9,-11,14.6,0,0,15.0,1,41:10,1,113.5,113.5,113.5,124.9,127.3,127.3,-11.4,-13.8,-13.8,0.000,0.0,0.0,0.041,0.029,0.036,7.1,7.2,0.375,0.349,0.133,0.136,104.17,103.19,85.99,103.19,-0.016,89,4.0,12.0,38,92,0.413,17,46,0.370,22,28,0.786,15,23,38,25,14.0,13,1,5,21,19,115,-16.0,111.3,112.7,127.7,129.7,-16.4,-17.0,0.658,1.79,17.2,0.321,0.649,0.452,0.137,0.505,0.551,102.9,101.5,84.58,102,0.403,1610612749,MIL,Milwaukee Bucks,50,83,0.602,16,32,0.500,15,24,0.625,11,33,44,30,20.0,11,5,1,19,21,131,16.0,127.7,129.7,111.3,112.7,16.4,17.0,0.600,1.50,20.8,0.351,0.679,0.548,0.198,0.699,0.700,102.9,101.5,84.58,101,0.597,G,NaN,NaN
149,NaN,2025-26,1629001,De'Anthony Melton,De'Anthony,1610612744,GSW,Golden State Warriors,22501142,2026-04-05T00:00:00,GSW vs. HOU,L,22.840000,2,4,0.500,2,4,0.50,0,0,0.000,0,1,1,4,1,0,1,0,0,0,6,-7,15.2,0,0,15.0,1,22:50,1,132.0,127.3,127.3,142.9,140.0,140.0,-11.0,-12.7,-12.7,0.200,4.0,44.4,0.000,0.083,0.031,11.1,11.1,0.750,0.750,0.100,0.108,90.91,93.52,77.93,93.52,0.064,44,2.0,4.0,42,84,0.500,14,40,0.350,18,20,0.900,6,25,31,34,10.0,7,3,7,19,16,116,-1.0,119.8,123.4,124.0,124.5,-4.2,-1.1,0.810,3.40,24.5,0.256,0.737,0.481,0.106,0.583,0.625,95.6,94.0,78.33,94,0.487,1610612745,HOU,Houston Rockets,44,80,0.550,13,29,0.448,16,19,0.842,8,30,38,30,14.0,4,7,3,16,19,117,1.0,124.0,124.5,119.8,123.4,4.2,1.1,0.682,2.14,22.4,0.263,0.744,0.519,0.149,0.631,0.662,95.6,94.0,78.33,94,0.513,G,PG,27.0
148,NaN,2025-26,1631288,Jamal Cain,Jamal,1610612753,ORL,Orlando Magic,22501138,2026-04-05T00:00:00,ORL @ NOP,W,27.621667,3,7,0.429,0,3,0.00,2,2,1.000,2,3,5,2,0,0,0,0,2,4,8,11,17.0,0,0,15.0,1,27:37,1,123.6,127.8,127.8,99.7,101.8,101.8,23.8,26.0,26.0,0.087,0.0,20.0,0.067,0.111,0.088,0.0,0.0,0.429,0.508,0.119,0.118,99.05,96.45,80.37,96.45,0.068,54,3.0,7.0,40,93,0.430,7,33,0.212,25,36,0.694,16,41,57,27,13.0,6,5,8,23,27,112,4.0,105.8,107.7,102.0,102.9,3.8,4.8,0.675,2.08,17.9,0.345,0.778,0.554,0.125,0.468,0.515,105.8,104.5,87.08,104,0.545,1610612740,NOP,New Orleans Pelicans,35,84,0.417,10,31,0.323,28,36,0.778,9,38,47,18,15.0,7,8,5,27,23,108,-4.0,102.0,102.9,105.8,107.7,-3.8,-4.8,0.514,1.20,13.4,0.222,0.655,0.446,0.143,0.476,0.541,105.8,104.5,87.08,105,0.455,NaN,SF,26.0
160,NaN,2025-26,1630644,Mac Mc

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260406_153717.json


,home_team,away_team,commence_time,bookmakers
0,Atlanta Hawks,New York Knicks,2026-04-06 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Orlando Magic,Detroit Pistons,2026-04-06 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Memphis Grizzlies,Cleveland Cavaliers,2026-04-07 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,San Antonio Spurs,Philadelphia 76ers,2026-04-07 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Denver Nuggets,Portland Trail Blazers,2026-04-07 01:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
BOOKMAKER = 'Underdog'
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs_pts = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_points')]
# lines_dfs_ast = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_assists')]
# lines_dfs_reb = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_rebounds')]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-06 15:36:02
US latest pull: 2026-04-06 15:37:18


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Jalen Brunson,Over,25.0,-137,2026-04-06,2026-04-06T22:35:56Z,2026-04-06 15:36:02
1,PrizePicks,player_points,Jalen Brunson,Under,25.0,-137,2026-04-06,2026-04-06T22:35:56Z,2026-04-06 15:36:02
2,PrizePicks,player_points,Jalen Johnson,Over,21.5,-137,2026-04-06,2026-04-06T22:35:56Z,2026-04-06 15:36:02
3,PrizePicks,player_points,Jalen Johnson,Under,21.5,-137,2026-04-06,2026-04-06T22:35:56Z,2026-04-06 15:36:02
4,PrizePicks,player_points,Nickeil Alexander-Walker,Over,19.5,-137,2026-04-06,2026-04-06T22:35:56Z,2026-04-06 15:36:02


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Paul Reed Jr: single positional indexer is out-of-bounds
[SKIP] G.G. Jackson: single positional indexer is out-of-bounds
[SKIP] Rayan Rupert: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Paul Reed Jr: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] G.G. Jackson: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90
0,Jalen Johnson,AST,28.62,34.20,40.26,0.0705,0.1685,0.2632,2.02,5.76,10.60
1,Jalen Brunson,AST,28.12,34.61,40.19,0.1120,0.2099,0.3081,3.15,7.27,12.38
2,Dyson Daniels,AST,22.65,30.39,36.94,0.0810,0.1548,0.2527,1.84,4.70,9.34
3,Josh Hart,AST,24.89,30.70,39.00,0.0768,0.1470,0.2397,1.91,4.51,9.35
4,Jonathan Kuminga,AST,16.05,22.49,27.82,0.0426,0.0960,0.1934,0.68,2.16,5.38
5,Daniss Jenkins,AST,19.60,28.98,38.31,0.0859,0.1742,0.2786,1.68,5.05,10.67
6,Jalen Suggs,AST,20.27,27.95,35.49,0.0730,0.1663,0.2664,1.48,4.65,9.46
7,Paolo Banchero,AST,28.61,34.54,40.59,0.0614,0.1455,0.2414,1.76,5.03,9.80
8,Anthony Black,AST,14.88,22.45,28.48,0.0535,0.1123,0.2311,0.80,2.52,6.58
9,Cedric Coward,AST,17.94,26.05,32.16,0.0338,0.1130,0.2023,0.61,2.94,6.51


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
91,Mikal Bridges,PTS,12.5,31.54,15.94,0.787,0.213
105,Tristan da Silva,PTS,9.5,26.98,10.28,0.636,0.364
80,De'Aaron Fox,REB,3.5,30.60,3.53,0.618,0.382
111,Cedric Coward,PTS,14.5,26.05,14.37,0.558,0.442
12,De'Aaron Fox,AST,5.5,30.60,5.48,0.626,0.374
144,Immanuel Quickley,PTS,13.5,27.16,15.61,0.681,0.319
26,Dennis Schröder,AST,7.5,26.34,5.30,0.353,0.647
97,Paolo Banchero,PTS,23.5,34.54,21.92,0.504,0.496
96,Zaccharie Risacher,PTS,4.5,20.11,8.81,0.943,0.057
44,Paolo Banchero,REB,7.5,34.54,7.38,0.604,0.396


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
144,Immanuel Quickley,PTS,13.5,27.16,15.61,0.681,0.319,PTS,Underdog,Miami Heat,1.5,239.5,113.4,11.0,104.40,1.0,-137.0,-137.0,0.578,0.578,13.7,12.5,5.33,0.2,-1.0,-0.038,0.515,0.485,-10.91,-16.10,0.4,0.4,0.60,0.69,31.85,3.80,0.19,0.02,18.67,3.0
139,Tyler Herro,PTS,21.5,32.12,20.29,0.512,0.488,PTS,Underdog,Toronto Raptors,-1.5,239.5,112.3,7.0,99.32,22.0,-137.0,-137.0,0.578,0.578,20.2,19.0,6.99,-1.3,-2.5,0.186,0.426,0.574,-26.31,-0.70,0.4,0.3,0.47,0.59,33.97,5.05,0.24,0.03,26.25,4.0
118,Stephon Castle,PTS,17.5,29.92,17.81,0.536,0.464,PTS,Underdog,Philadelphia 76ers,-8.5,237.0,114.9,17.0,100.32,16.0,100.0,-114.0,0.500,0.533,17.3,19.5,5.83,-0.2,2.0,0.034,0.486,0.514,-2.80,-3.51,0.8,0.6,0.60,0.41,30.63,6.00,0.23,0.04,16.33,3.0
126,Deni Avdija,PTS,25.5,30.48,17.75,0.187,0.813,PTS,Underdog,Denver Nuggets,7.5,236.0,116.0,21.0,99.47,20.0,-115.0,-107.0,0.535,0.517,22.8,21.5,4.85,-2.7,-4.0,0.557,0.289,0.711,-45.97,37.55,0.4,0.3,0.20,0.30,31.68,5.56,0.30,0.03,20.86,7.0
141,Norman Powell,PTS,17.5,28.59,17.94,0.600,0.400,PTS,Underdog,Toronto Raptors,-1.5,239.5,112.3,7.0,99.32,22.0,-137.0,-137.0,0.578,0.578,18.1,19.5,6.45,0.6,2.0,-0.093,0.537,0.463,-7.10,-19.90,0.6,0.6,0.67,0.76,27.49,6.80,0.25,0.05,20.33,3.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
49,Evan Mobley,REB,9.0,30.32,9.61,0.559,0.354,REB,PrizePicks,Memphis Grizzlies,-13.5,231.5,117.8,24.0,101.43,9.0,-137.0,-137.0,0.578,0.578,10.3,10.0,4.16,1.3,1.0,-0.313,0.623,0.377,7.77,-34.78,0.6,0.6,0.47,0.49,31.17,4.18,0.21,0.06,12.33,3.0
3,Josh Hart,AST,4.5,30.70,4.51,0.584,0.416,AST,PrizePicks,Atlanta Hawks,1.5,227.5,112.7,9.0,102.54,5.0,-113.0,-105.0,0.531,0.512,3.9,4.0,1.20,-0.6,-0.5,0.500,0.309,0.691,-41.75,34.91,0.2,0.3,0.33,0.62,31.35,5.20,0.15,0.06,7.20,5.0
137,Bruce Brown,PTS,7.5,23.58,8.81,0.747,0.253,PTS,PrizePicks,Portland Trail Blazers,-7.5,236.0,113.5,13.0,101.85,7.0,-137.0,-137.0,0.578,0.578,9.1,10.0,4.31,2.1,3.0,-0.487,0.687,0.313,18.85,-45.85,0.4,0.6,0.47,0.48,20.70,4.90,0.17,0.05,8.00,3.0
61,Donovan Clingan,REB,11.5,25.89,10.89,0.472,0.528,REB,PrizePicks,Denver Nuggets,7.5,236.0,116.0,21.0,99.47,20.0,-137.0,-137.0,0.578,0.578,11.4,12.5,4.27,0.4,1.5,-0.094,0.537,0.463,-7.10,-19.90,0.4,0.6,0.60,0.36,25.84,3.61,0.19,0.07,11.00,6.0
107,Evan Mobley,PTS,21.5,30.32,17.48,0.331,0.669,PTS,PrizePicks,Memphis Grizzlies,-13.5,231.5,117.8,24.0,101.43,9.0,-110.0,102.0,0.524,0.495,19.1,18.5,8.79,-2.4,-3.0,0.273,0.392,0.608,-25.16,22.82,0.4,0.4,0.40,0.32,31.17,4.18,0.21,0.06,23.00,3.0


In [11]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
6,Jalen Suggs,AST,5.5,27.95,4.65,0.442,0.558,AST,PrizePicks,Detroit Pistons,1.5,222.5,108.6,2.0,99.91,19.0,-115.0,102.0,0.535,0.495,6.0,6.0,2.45,0.5,0.5,-0.204,0.581,0.419,8.62,-15.36,0.8,0.6,0.53,0.38,30.57,4.87,0.21,0.04,2.83,6.0
128,Jrue Holiday,PTS,17.5,31.55,15.16,0.505,0.495,PTS,PrizePicks,Denver Nuggets,7.5,236.0,116.0,21.0,99.47,20.0,-110.0,-105.0,0.524,0.512,16.2,12.5,7.57,-1.3,-5.0,0.172,0.432,0.568,-17.53,10.90,0.6,0.3,0.40,0.26,29.94,4.60,0.22,0.06,14.50,4.0
50,Cedric Coward,REB,5.0,26.05,5.54,0.565,0.317,REB,PrizePicks,Cleveland Cavaliers,13.5,231.5,114.0,14.0,100.60,13.0,-137.0,-137.0,0.578,0.578,4.5,3.0,4.97,-0.5,-2.0,0.101,0.460,0.540,-20.42,-6.58,0.2,0.3,0.40,0.52,25.24,1.62,0.22,0.07,8.00,1.0
49,Evan Mobley,REB,9.0,30.32,9.61,0.559,0.354,REB,PrizePicks,Memphis Grizzlies,-13.5,231.5,117.8,24.0,101.43,9.0,-137.0,-137.0,0.578,0.578,10.3,10.0,4.16,1.3,1.0,-0.313,0.623,0.377,7.77,-34.78,0.6,0.6,0.47,0.49,31.17,4.18,0.21,0.06,12.33,3.0
52,Victor Wembanyama,REB,13.5,30.21,11.42,0.390,0.610,REB,PrizePicks,Philadelphia 76ers,-8.5,237.0,114.9,17.0,100.32,16.0,-137.0,-137.0,0.578,0.578,13.8,15.0,3.58,0.8,2.0,-0.223,0.588,0.412,1.72,-28.73,1.0,0.6,0.47,0.30,29.98,5.10,0.32,0.04,8.50,2.0


### Get top EVs

In [12]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 108  |  Pairs: 298  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [13]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 35  |  Pairs: 13  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
